# Average Audio Files

Load each experiment's `sync.csv`, then average microphone pairs for a range of experiments.

In [4]:
import ast
import platform
import re
import shutil
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.io import wavfile

REPO_ROOT = Path("/mnt/home/gginosar/repos/gerbil_vocalization_analysis")
if (REPO_ROOT / "vocalization_analysis").exists() and str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from vocalization_analysis.audio_processing_config import (
    detect_raw_naming_scheme,
    get_channel_mapping,
    get_experiment_month,
    should_skip_experiment,
)

if platform.system() == "Windows":
    base_raw = Path(r"\\sanesstorage.cns.nyu.edu\archive\ginosar\Raw_data")
    base_processed = Path(r"\\sanesstorage.cns.nyu.edu\archive\ginosar\Processed_data")
else:
    base_raw = Path("/mnt/home/neurostatslab/ceph/saneslab_data/big_setup/")
    base_processed = Path("/mnt/home/neurostatslab/ceph/saneslab_data/gily_data/Processed_data")

MAX_STANDARD_WAV_SIZE_BYTES = (4 * 1024**3) - 1

print("Raw data base:", base_raw)
print("Processed data base:", base_processed)


Raw data base: /mnt/home/neurostatslab/ceph/saneslab_data/big_setup
Processed data base: /mnt/home/neurostatslab/ceph/saneslab_data/gily_data/Processed_data


In [2]:
def build_experiment_paths(exp):
    month_folder = get_experiment_month(exp)
    experiment_root = base_raw / f"experiment_{exp}"
    folder_path_raw_wavs = experiment_root / "concatenated_data_cam_mic_sync"
    folder_path_sync = folder_path_raw_wavs
    folder_path_averaged_wavs = base_processed / "Audio" / month_folder / str(exp) / "Averaged_wavs_w_annotations"
    folder_path_audio = base_processed / "Audio" / month_folder / str(exp)
    return {
        "month_folder": month_folder,
        "experiment_root": experiment_root,
        "raw_wavs": folder_path_raw_wavs,
        "sync": folder_path_sync,
        "averaged_wavs": folder_path_averaged_wavs,
        "audio": folder_path_audio,
    }


def load_sync_file(exp, folder_path_sync):
    sync_path = Path(folder_path_sync) / "sync.csv"
    if not sync_path.exists():
        raise FileNotFoundError(f"Sync file not found: {sync_path}")

    sync_df = pd.read_csv(sync_path)
    print(list(sync_df.columns))

    sync_df["timestamp"] = sync_df["timestamp"].apply(ast.literal_eval)
    sync_df["start_time"] = pd.to_datetime(sync_df["timestamp"].str[0])
    start_time_experiment = sync_df.iloc[0]["start_time"]
    print(f"Experiment {exp} started at: {start_time_experiment}")

    return start_time_experiment, sync_df


def collect_chunk_paths(input_folder, file_num, raw_naming_scheme):
    if raw_naming_scheme == "legacy":
        suffix = f"_{file_num}"
    else:
        suffix = f"_{file_num:03d}"

    chunk_paths = [
        path for path in input_folder.iterdir()
        if path.is_file() and path.suffix.lower() in {".wav", ".mp4"} and path.stem.endswith(suffix)
    ]
    return sorted(chunk_paths)


def find_oversized_chunk_files(chunk_paths):
    return [path for path in chunk_paths if path.suffix.lower() == ".wav" and path.stat().st_size > MAX_STANDARD_WAV_SIZE_BYTES]


def remove_terminal_problem_chunk(input_folder, file_num, raw_naming_scheme, problem_paths):
    chunk_paths = collect_chunk_paths(input_folder, file_num, raw_naming_scheme)
    print(
        f"Removing terminal problematic chunk {file_num:03d}; oversized WAV files: "
        f"{', '.join(path.name for path in problem_paths)}"
    )
    for path in chunk_paths:
        path.unlink()
        print(f"  Deleted {path.name}")


def average_microphone_pairs(exp, input_folder, output_folder):
    input_folder = Path(input_folder)
    output_folder = Path(output_folder)
    output_folder.mkdir(parents=True, exist_ok=True)

    channel_mapping = get_channel_mapping(exp)
    raw_naming_scheme = detect_raw_naming_scheme(exp, input_folder)

    print(f"\n--- Processing experiment {exp} ---")
    print(f"Reading raw WAV files from: {input_folder}")
    print(f"Saving averaged WAV files to: {output_folder}")
    print(f"Raw naming scheme: {raw_naming_scheme}")

    for virtual_ch, real_pair in channel_mapping.items():
        print(f"  virtual channel {virtual_ch} <- real channels {real_pair}")

    source_channels = sorted({channel for pair in channel_mapping.values() for channel in pair})
    file_nums = set()
    for source_channel in source_channels:
        if raw_naming_scheme == "legacy":
            pattern = re.compile(rf"channel_{source_channel}_(\d+)\.wav$")
        else:
            pattern = re.compile(rf"channel_{source_channel:02d}_file_(\d+)\.wav$")

        for file_path in input_folder.iterdir():
            match = pattern.match(file_path.name)
            if match:
                file_nums.add(int(match.group(1)))

    file_nums = sorted(file_nums)
    print(f"Found {len(file_nums)} file chunks")

    n_saved = 0
    n_missing = 0
    n_shape_mismatch = 0
    n_rate_mismatch = 0

    dropped_terminal_chunks = []

    for index, file_num in enumerate(file_nums):
        file_num_str = f"{file_num:03d}"
        chunk_paths = collect_chunk_paths(input_folder, file_num, raw_naming_scheme)
        oversized_chunk_files = find_oversized_chunk_files(chunk_paths)
        if oversized_chunk_files:
            is_last_chunk = index == len(file_nums) - 1
            if not is_last_chunk:
                oversized_names = ', '.join(path.name for path in oversized_chunk_files)
                raise RuntimeError(
                    f"Problematic non-terminal chunk {file_num_str} in experiment {exp}; "
                    f"oversized WAV files: {oversized_names}"
                )

            remove_terminal_problem_chunk(input_folder, file_num, raw_naming_scheme, oversized_chunk_files)
            dropped_terminal_chunks.append(file_num)
            continue

        for virtual_ch, real_pair in channel_mapping.items():
            ch1, ch2 = real_pair
            if raw_naming_scheme == "legacy":
                path1 = input_folder / f"channel_{ch1}_{file_num}.wav"
                path2 = input_folder / f"channel_{ch2}_{file_num}.wav"
            else:
                path1 = input_folder / f"channel_{ch1:02d}_file_{file_num_str}.wav"
                path2 = input_folder / f"channel_{ch2:02d}_file_{file_num_str}.wav"

            if not path1.exists() or not path2.exists():
                print("  Skipping because one or both files are missing")
                print(f"  Expected file: {path1.name}")
                print(f"  Expected file: {path2.name}")
                n_missing += 1
                continue

            rate1, data1 = wavfile.read(path1)
            rate2, data2 = wavfile.read(path2)

            if rate1 != rate2:
                print(f"  Skipping because sample rates differ: {rate1} vs {rate2}")
                n_rate_mismatch += 1
                continue

            if data1.shape != data2.shape:
                print(f"  Skipping because shapes differ: {data1.shape} vs {data2.shape}")
                n_shape_mismatch += 1
                continue

            avg_data = (data1.astype(np.float32, copy=False) + data2.astype(np.float32, copy=False)) / 2.0
            out_path = output_folder / f"channel_{virtual_ch}_file_{file_num_str}.wav"
            wavfile.write(out_path, rate1, avg_data)
            n_saved += 1

    print("\n--- Done ---")
    print(f"Saved files: {n_saved}")
    print(f"Skipped because of missing files: {n_missing}")
    print(f"Skipped because of shape mismatch: {n_shape_mismatch}")
    print(f"Skipped because of sample rate mismatch: {n_rate_mismatch}")
    print(f"Dropped terminal problematic chunks: {dropped_terminal_chunks}")

    return {
        "saved_files": n_saved,
        "missing_pairs": n_missing,
        "shape_mismatch": n_shape_mismatch,
        "rate_mismatch": n_rate_mismatch,
        "dropped_terminal_chunks": len(dropped_terminal_chunks),
    }


def copy_experiment_log_file(exp, experiment_root, destination_folder):
    experiment_root = Path(experiment_root)
    destination_folder = Path(destination_folder)
    destination_folder.mkdir(parents=True, exist_ok=True)

    log_files = sorted(experiment_root.glob(f"experiment_{exp}_log*.txt"))
    if not log_files:
        print(f"Warning: no log txt file found in {experiment_root}")
        return None

    log_path = log_files[0]
    destination_path = destination_folder / log_path.name
    shutil.copy2(log_path, destination_path)
    print(f"Copied log file to: {destination_path}")
    return destination_path


In [5]:
# Set the experiment range here.
start_exp = 560
end_exp = 560
#experiment_ids = list(range(start_exp, end_exp + 1))

# Optional: use a manual list instead.
experiment_ids = [504,527,546,557,566]

stop_on_error = False
results = []

for exp in experiment_ids:
    print(f"\n{'=' * 80}")
    print(f"Starting experiment {exp}")

    try:
        if should_skip_experiment(exp):
            print(f"Skipping experiment {exp} by design")
            results.append({"exp": exp, "skipped": True, "reason": "configured skip"})
            continue

        paths = build_experiment_paths(exp)
        print("Raw WAV folder:", paths["raw_wavs"])
        print("Sync folder:", paths["sync"])
        print("Output folder:", paths["averaged_wavs"])
        print("Processed experiment folder:", paths["audio"])

        start_time_experiment, sync_df = load_sync_file(exp, paths["sync"])
        log_copy_path = copy_experiment_log_file(exp, paths["experiment_root"], paths["audio"])
        summary = average_microphone_pairs(exp, paths["raw_wavs"], paths["averaged_wavs"])

        results.append({
            "exp": exp,
            "month_folder": paths["month_folder"],
            "start_time_experiment": start_time_experiment,
            "sync_rows": len(sync_df),
            "copied_log_file": str(log_copy_path),
            **summary,
        })
    except Exception as exc:
        print(f"Failed for experiment {exp}: {exc}")
        results.append({"exp": exp, "error": str(exc)})
        if stop_on_error:
            raise

results_df = pd.DataFrame(results)
results_df



Starting experiment 504
Raw WAV folder: /mnt/home/neurostatslab/ceph/saneslab_data/big_setup/experiment_504/concatenated_data_cam_mic_sync
Sync folder: /mnt/home/neurostatslab/ceph/saneslab_data/big_setup/experiment_504/concatenated_data_cam_mic_sync
Output folder: /mnt/home/neurostatslab/ceph/saneslab_data/gily_data/Processed_data/Audio/2026_02/504/Averaged_wavs_w_annotations
Processed experiment folder: /mnt/home/neurostatslab/ceph/saneslab_data/gily_data/Processed_data/Audio/2026_02/504
['video', 'audio', 'timestamp']
Experiment 504 started at: 2026-02-21 18:19:17
Copied log file to: /mnt/home/neurostatslab/ceph/saneslab_data/gily_data/Processed_data/Audio/2026_02/504/experiment_504_log_2026-02-21_18-18-57.txt

--- Processing experiment 504 ---
Reading raw WAV files from: /mnt/home/neurostatslab/ceph/saneslab_data/big_setup/experiment_504/concatenated_data_cam_mic_sync
Saving averaged WAV files to: /mnt/home/neurostatslab/ceph/saneslab_data/gily_data/Processed_data/Audio/2026_02/50

,exp,month_folder,start_time_experiment,sync_rows,copied_log_file,saved_files,missing_pairs,shape_mismatch,rate_mismatch,dropped_terminal_chunks,error
0,504,2026_02,2026-02-21 18:19:17,208.0,/mnt/home/neurostatslab/ceph/saneslab_data/gil...,624.0,0.0,0.0,0.0,0.0,NaN
1,527,2026_02,2026-03-02 09:50:19,26.0,/mnt/home/neurostatslab/ceph/saneslab_data/gil...,78.0,0.0,0.0,0.0,0.0,NaN
2,546,2026_02,2026-03-05 12:03:52,30.0,/mnt/home/neurostatslab/ceph/saneslab_data/gil...,90.0,0.0,0.0,0.0,0.0,NaN
3,557,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,[Errno 13] Permission denied: '/mnt/home/neuro...
4,566,2026_02,2026-03-11 13:15:57,6.0,/mnt/home/neurostatslab/ceph/saneslab_data/gil...,18.0,0.0,0.0,0.0,0.0,NaN


In [15]:
# Copy missing raw log txt files into Processed_data/Audio for supported experiments >= 272.

log_copy_results = []

for experiment_root in sorted(base_raw.glob("experiment_*")):
    if not experiment_root.is_dir():
        continue

    try:
        exp = int(experiment_root.name.split("_")[1])
    except (IndexError, ValueError):
        continue

    if exp < 272 or should_skip_experiment(exp):
        continue

    try:
        month_folder = get_experiment_month(exp)
    except ValueError:
        continue

    processed_audio_folder = base_processed / "Audio" / month_folder / str(exp)
    raw_log_files = sorted(experiment_root.glob(f"experiment_{exp}_log*.txt"))

    if not raw_log_files:
        print(f"Experiment {exp}: no raw log txt file found")
        log_copy_results.append({"exp": exp, "status": "no_raw_log_found"})
        continue

    processed_audio_folder.mkdir(parents=True, exist_ok=True)

    copied_files = []
    for raw_log_path in raw_log_files:
        destination_path = processed_audio_folder / raw_log_path.name
        if destination_path.exists():
            continue
        shutil.copy2(raw_log_path, destination_path)
        copied_files.append(destination_path.name)

    if copied_files:
        print(f"Experiment {exp}: copied {len(copied_files)} log file(s)")
        log_copy_results.append({"exp": exp, "status": "copied", "copied_files": copied_files})
    else:
        log_copy_results.append({"exp": exp, "status": "already_present"})

log_copy_results_df = pd.DataFrame(log_copy_results)
log_copy_results_df


Experiment 272: copied 1 log file(s)
Experiment 275: copied 1 log file(s)
Experiment 276: copied 1 log file(s)
Experiment 277: copied 1 log file(s)
Experiment 278: copied 1 log file(s)
Experiment 491: copied 1 log file(s)
Experiment 492: copied 1 log file(s)
Experiment 493: copied 1 log file(s)
Experiment 494: copied 1 log file(s)
Experiment 506: copied 1 log file(s)
Experiment 508: copied 1 log file(s)
Experiment 509: copied 1 log file(s)
Experiment 510: copied 1 log file(s)
Experiment 511: copied 1 log file(s)
Experiment 512: copied 1 log file(s)
Experiment 527: no raw log txt file found
Experiment 546: no raw log txt file found
Experiment 547: copied 1 log file(s)
Experiment 557: copied 1 log file(s)
Experiment 558: copied 1 log file(s)
Experiment 559: copied 1 log file(s)
Experiment 560: copied 1 log file(s)
Experiment 561: copied 1 log file(s)
Experiment 562: copied 1 log file(s)
Experiment 563: copied 1 log file(s)
Experiment 564: copied 1 log file(s)
Experiment 565: copied 1 log

,exp,status,copied_files
0,272,copied,[experiment_272_log_2025-07-03_16-17-37.txt]
1,273,already_present,NaN
2,275,copied,[experiment_275_log_2025-07-07_11-37-23.txt]
3,276,copied,[experiment_276_log_2025-07-08_16-15-04.txt]
4,277,copied,[experiment_277_log_2025-07-09_09-59-01.txt]
...,...,...,...
94,575,copied,[experiment_575_log_2026-03-26_17-56-13.txt]
95,576,copied,[experiment_576_log_2026-03-26_18-29-03.txt]
96,577,copied,[experiment_577_log_2026-03-26_19-07-29.txt]
97,578,copied,[experiment_578_log_2026-03-26_19-44-58.txt]
